In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so
import anndata

In [ ]:
var = pd.read_csv('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/Synapse_Data/Synapse_Data/filtered_gene_row_names.txt', sep='\t')
column_name_mapping = {'FO538757.2': 'gene_ids'}
var.rename(columns=column_name_mapping, inplace=True)

In [ ]:
var['gene_ids_duplicate'] = var.loc[:, 'gene_ids']
column_name_mapping = {'gene_ids_duplicate': 'Feature_ids'}
var.rename(columns=column_name_mapping, inplace=True)

In [ ]:
var

In [ ]:
var.to_csv('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/filtered_gene_row_names.txt')

In [ ]:
obs1 = pd.read_csv('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/Synapse_Data/Synapse_Data/filtered_column_metadata.txt', sep='\t')

In [ ]:
obs2 = pd.read_csv('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/ROSMAP_Metadata/ROSMAP_Metadata/ROSMAP_clinical.csv')


In [ ]:
# Initialize a new column 'Classification' with NA values
obs2['disease'] = None

# Iterate over rows and apply the classification logic
for i in range(len(obs2)):
    # Check if CERAD, braaksc, and cts_mmse30_lv are not NaN
    if pd.notna(obs2.loc[i, 'ceradsc']) and pd.notna(obs2.loc[i, 'braaksc']) and pd.notna(obs2.loc[i, 'cts_mmse30_lv']):
        
        # Check for conditions to classify as Control, AsymAD, AD, or Exclude
        if ((obs2.loc[i, 'ceradsc'] in [4, 3] and obs2.loc[i, 'braaksc'] in [0, 1, 2]) or 
            (obs2.loc[i, 'ceradsc'] == 4 and obs2.loc[i, 'braaksc'] == 3)) and obs2.loc[i, 'cts_mmse30_lv'] >= 24:
            obs2.loc[i, 'disease'] = 'Control'
        elif (obs2.loc[i, 'ceradsc'] in [3, 2, 1] and obs2.loc[i, 'braaksc'] in [3, 4, 5, 6] and 
              obs2.loc[i, 'cts_mmse30_lv'] >= 24):
            obs2.loc[i, 'disease'] = 'AsymAD'
        elif (obs2.loc[i, 'ceradsc'] in [2, 1] and obs2.loc[i, 'braaksc'] in [3, 4, 5, 6] and 
              obs2.loc[i, 'cts_mmse30_lv'] < 24):
            obs2.loc[i, 'disease'] = 'AD'
        else:
            obs2.loc[i, 'disease'] = 'Exclude'

# Set 'NA' for any remaining missing classifications
obs2['disease'].fillna('NA', inplace=True)

In [ ]:
obs1.head(3)

In [ ]:
obs2.head(3)

In [ ]:
# Assuming 'disease' is the name of your column
disease_counts = obs2['disease'].value_counts()

# Display the result
print(disease_counts)


In [ ]:
obs3=obs1.merge(obs2, on="projid", how="left" )

In [ ]:
column_name_mapping = {'broad.cell.type': 'cell_type','msex': 'sex'}

In [ ]:
obs3.rename(columns=column_name_mapping, inplace=True)

In [ ]:
obs3 = obs3.replace(np.nan, ' ')

In [ ]:
obs3.head(3)

In [ ]:
obs3.to_csv('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/filtered_column_metadata.txt')

In [ ]:
adata = sc.read('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/Synapse_Data/Synapse_Data/filtered_count_matrix.mtx',cache=True).transpose()

In [ ]:
adata.var = var
adata.obs = obs3

In [ ]:
adata.obs = adata.obs.astype(str)
adata.var = adata.var.astype(str)

In [ ]:
adata.var.set_index('gene_ids', inplace=True)
adata.var_names_make_unique()

In [ ]:
adata.write('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/ADP1.h5ad')

In [ ]:
if 'cell_type' in adata.obs.columns:
    cell_type_counts = adata.obs['cell_type'].value_counts()
    print("Cell type counts:")
    print(cell_type_counts)
    
    # Get the total count of all cells
    total_cell_count = cell_type_counts.sum()
    print(f"Total number of cells: {total_cell_count}")
else:
    print("Cell type information not found in 'cell_type' column.")

In [ ]:
adata

In [ ]:
sc.logging.print_header()
sc.settings.set_figure_params(facecolor="white")

In [ ]:
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata

In [ ]:
sc.pp.scale(adata, max_value=10)

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=50)

In [ ]:
sc.tl.tsne(adata)

In [ ]:
sc.tl.louvain(adata)

In [ ]:
adata.write('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/ADP3.h5ad')

In [ ]:
marker_genes  = ["FLT1","CLDN5","AQP4","GFAP", "C3","CD74","CSF1R","PDGFRA","VCAN","CSPG4","MBP","MOBP","PLP1","GAD1","GAD2","NRGN","SLC17A7","CAMK2A","SYT1","GRIN1","SNAP25"]

In [ ]:
plt.rcParams['font.size'] = 16
fig1 = sc.pl.dotplot(adata, marker_genes, groupby="cell_type",gene_symbols="gene_ids", dendrogram=True, return_fig=True)
fig1.savefig('dotplot1.pdf', dpi=300)
plt.close(fig1)

In [ ]:
risk_genes = ['MICU3', 'SLC7A2','RBM14','RBM4','RBM14-RBM4','SLC12A5','MRAS','GALNT8','KCNA6', 'STK24','PRDM4','TRIM23','VPS13C', 'BMPR1B','FNTB','CHURC1-FNTB','RAB15', 'LMO7']

In [ ]:
bdata = adata[:, adata.var.gene_ids.isin(risk_genes)]

In [ ]:
bdata.var

In [ ]:
bdata.write('C:/Users/mdhdu/AppData/Local/Programs/Python310/Scripts/cmd/data/BDP3.h5ad')

In [ ]:
# Set the font size for the plot
plt.rcParams['font.size'] = 16

# Create the t-SNE plot and return the figure
fig = sc.pl.tsne(adata, color="cell_type", legend_fontsize=14, return_fig=True)

# Save the plot to a PDF file with high resolution (300 dpi)
fig.savefig('tsne_plot1.pdf', dpi=300, bbox_inches='tight')

# Close the figure after saving
plt.close(fig)

In [ ]:
# Define the group order, excluding 'Exclude'
group_order = ["Control", "AsymAD", "AD"]

# Filter the data to remove 'Exclude' from the 'disease' column
adata_filtered = adata[adata.obs['disease'] != 'Exclude'].copy()

# Set the font size for the plot
plt.rcParams['font.size'] = 16

# Create the t-SNE plot using the filtered data and return the figure
fig = sc.pl.tsne(adata_filtered, color="disease", legend_fontsize=14, return_fig=True)

# Save the plot to a PDF file with high resolution (300 dpi)
fig.savefig('tsne_plot2.pdf', dpi=300, bbox_inches='tight')

# Close the figure after saving
plt.close(fig)

In [ ]:
plt.rcParams['font.size'] = 42
plt.figure(figsize=(10, 10))
sc.pl.tsne(adata, 
           color=['SLC7A2', 'STK24', 'BMPR1B','MRAS','LMO7','TRIM23','RAB15', 'GALNT8', 'RBM14-RBM4','CHURC1-FNTB', 'RBM4','FNTB', 'PRDM4', 'SLC12A5','MICU3','VPS13C'],
           cmap='Reds',  # Change the color map to "Reds" (white to red)
           legend_fontsize=42,  # Increase the legend font size
           gene_symbols="gene_ids",
           show=False)  # Prevent the figure from showing immediately
plt.savefig('tsne_plot3.pdf', dpi=300, bbox_inches='tight')  # Save as PNG with 300 dpi

In [ ]:
# Create dataframe1: cell count for each cell type and disease condition
cell_counts = bdata.obs.groupby(['cell_type', 'disease']).size().unstack(fill_value=0)
# Create dataframe2: sample count for each cell type and disease condition
sample_counts = bdata.obs.groupby(['cell_type', 'disease'])['projid'].nunique().unstack(fill_value=0)
# Save both dataframes to an Excel file
with pd.ExcelWriter('cell_and_sample_counts.xlsx') as writer:
    cell_counts.to_excel(writer, sheet_name='Cell_Counts')
    sample_counts.to_excel(writer, sheet_name='Sample_Counts')

print("DataFrames have been saved to 'cell_and_sample_counts.xlsx'.")